In [1]:
import random

# --- CONFIGURATION & SIMULATION SETUP ---
# Average number of vehicles arriving per minute in each direction
TRAFFIC_DENSITIES = {
    'North': 45,
    'South': 40,
    'East': 15,
    'West': 20
}

TOTAL_CYCLE_TIME = 120  # The total green time (seconds) to distribute across directions
POPULATION_SIZE = 50    # Number of candidate solutions in each generation
GENERATIONS = 100       # Number of iterations to find the best solution
MUTATION_RATE = 0.15    # Probability of mutating a solution

def calculate_waiting_time(green_times):
    """
    Fitness Function: Calculates the total waiting time for a given set of green times.
    Higher vehicle density + shorter green time = significantly higher waiting time.
    """
    total_waiting_time = 0

    for direction, density in TRAFFIC_DENSITIES.items():
        green_time = green_times[direction]

        # If a busy road gets too little green time, waiting time escalates exponentially
        if green_time == 0:
            waiting_time = density * 100
        else:
            # Formula modeling queue delays: higher density creates more wait time,
            # mitigated proportionally by green light allocation.
            waiting_time = (density ** 2) / (green_time * 0.5)

        total_waiting_time += waiting_time

    return total_waiting_time

# --- GENETIC ALGORITHM COMPONENTS ---

def create_individual():
    """Generates a random valid solution where green times sum up to TOTAL_CYCLE_TIME."""
    directions = list(TRAFFIC_DENSITIES.keys())
    # Generate random split points
    splits = sorted([random.randint(0, TOTAL_CYCLE_TIME) for _ in range(len(directions) - 1)])

    # Calculate durations between split points
    durations = [splits[0]]
    for i in range(1, len(splits)):
        durations.append(splits[i] - splits[i-1])
    durations.append(TOTAL_CYCLE_TIME - splits[-1])

    return dict(zip(directions, durations))

def create_population():
    return [create_individual() for _ in range(POPULATION_SIZE)]

def crossover(parent1, parent2):
    """Blends two parent distributions and normalizes them to preserve TOTAL_CYCLE_TIME."""
    child_raw = {}
    for direction in TRAFFIC_DENSITIES.keys():
        # Take the average of both parents
        child_raw[direction] = (parent1[direction] + parent2[direction]) / 2

    # Normalize to ensure the sum equals TOTAL_CYCLE_TIME perfectly
    total_raw = sum(child_raw.values())
    child = {}
    for direction, val in child_raw.items():
        child[direction] = round((val / total_raw) * TOTAL_CYCLE_TIME)

    # Correct rounding errors to lock exactly onto TOTAL_CYCLE_TIME
    difference = TOTAL_CYCLE_TIME - sum(child.values())
    child[random.choice(list(TRAFFIC_DENSITIES.keys()))] += difference

    return child

def mutate(individual):
    """Slightly shifts time from one random direction to another."""
    if random.random() < MUTATION_RATE:
        dirs = list(TRAFFIC_DENSITIES.keys())
        giver, receiver = random.sample(dirs, 2)

        # Shift a random slice of time (up to 10 seconds)
        shift_amount = random.randint(1, min(10, max(1, individual[giver])))
        individual[giver] -= shift_amount
        individual[receiver] += shift_amount

    return individual

# --- MAIN EXECUTION ENGINE ---

def run_genetic_algorithm():
    # Step 1: Initialize population
    population = create_population()

    print(f"Optimizing traffic signals for densities: {TRAFFIC_DENSITIES}")
    print(f"Total cycle time allocation budget: {TOTAL_CYCLE_TIME} seconds\n Running...")

    for generation in range(GENERATIONS):
        # Step 2: Evaluate fitness (Lower waiting time = better fitness)
        # We sort by waiting time ascending
        population = sorted(population, key=lambda ind: calculate_waiting_time(ind))

        # Elitism: Keep the top 2 solutions unchanged
        new_population = population[:2]

        # Step 3: Selection, Crossover & Mutation to fill out the rest of the generation
        while len(new_population) < POPULATION_SIZE:
            # Tournament-style selection from the top 50% performers
            p1 = random.choice(population[:POPULATION_SIZE // 2])
            p2 = random.choice(population[:POPULATION_SIZE // 2])

            child = crossover(p1, p2)
            child = mutate(child)
            new_population.append(child)

        population = new_population

    # Step 4: Extract the ultimate best solution
    best_solution = sorted(population, key=lambda ind: calculate_waiting_time(ind))[0]
    best_score = calculate_waiting_time(best_solution)

    print("\n--- OPTIMIZATION RESULTS ---")
    print(f"Optimal Green Light Durations:")
    for direction, seconds in best_solution.items():
        print(f" 🟢 {direction} Junction: {seconds} seconds (Density: {TRAFFIC_DENSITIES[direction]} vehicles/min)")

    print(f"\nSum Check: {sum(best_solution.values())} seconds distributed.")
    print(f"Minimized Traffic Congestion Score (Lower is better): {round(best_score, 2)}")

if __name__ == "__main__":
    run_genetic_algorithm()


Optimizing traffic signals for densities: {'North': 45, 'South': 40, 'East': 15, 'West': 20}
Total cycle time allocation budget: 120 seconds
 Running...

--- OPTIMIZATION RESULTS ---
Optimal Green Light Durations:
 🟢 North Junction: 45 seconds (Density: 45 vehicles/min)
 🟢 South Junction: 40 seconds (Density: 40 vehicles/min)
 🟢 East Junction: 15 seconds (Density: 15 vehicles/min)
 🟢 West Junction: 20 seconds (Density: 20 vehicles/min)

Sum Check: 120 seconds distributed.
Minimized Traffic Congestion Score (Lower is better): 240.0


### Pseudo Code: Traffic Light Optimization using Genetic Algorithm

```text
ALGORITHM TrafficLightGeneticAlgorithm

    Define TRAFFIC_DENSITIES (North, South, East, West)
    Define TOTAL_CYCLE_TIME = 120
    Define POPULATION_SIZE = 50
    Define GENERATIONS = 100
    Define MUTATION_RATE = 0.15

    FUNCTION CalculateWaitingTime(individual):
        total_waiting_time = 0
        FOR EACH direction, density IN TRAFFIC_DENSITIES:
            green_time = individual[direction]
            IF green_time == 0 THEN
                waiting_time = density * 100
            ELSE
                waiting_time = (density^2) / (green_time * 0.5)
            END IF
            total_waiting_time = total_waiting_time + waiting_time
        END FOR
        RETURN total_waiting_time
    END FUNCTION

    FUNCTION RunOptimization():
        population = CreateInitialPopulation(POPULATION_SIZE, TOTAL_CYCLE_TIME)

        FOR generation = 1 TO GENERATIONS:
            Sort population by CalculateWaitingTime(individual) ASCENDING

            new_population = [ population[0], population[1] ]

            WHILE Size(new_population) < POPULATION_SIZE:
                parent1 = RandomChoice(population[0 TO POPULATION_SIZE / 2])
                parent2 = RandomChoice(population[0 TO POPULATION_SIZE / 2])

                child = Crossover(parent1, parent2)
                child = Mutate(child, MUTATION_RATE)

                Append child to new_population
            END WHILE

            population = new_population
        END FOR

        best_solution = population[0]
        Print best_solution and fitness score
    END FUNCTION
END ALGORITHM
```